# Project Overview — BGC Donor Data Consolidation

## Problem
Boys & Girls Club (BGC) donor data is fragmented across multiple sources (spreadsheets and paper records).  
This makes it difficult to get a unified donor view for communication, recognition, reporting, and trend analysis.

## Primary Goal
Create a **single source of truth** for donor information by consolidating scattered data into a unified, usable dataset.

## Project Outputs (Planned)
- A cleaned, unified donor dataset (single source of truth)
- A donor-level view (one row per donor) and a donations view (one row per donation), if applicable
- A Streamlit dashboard for internal visualization and exploration
- A data quality summary + documentation (data dictionary + cleaning notes)

## Potential Research Areas (Team Tracks)
- **Data Consolidation:** Build a unified dataset + Streamlit visualization dashboard
- **Donor Prediction:** Identify characteristics of potential donors or predict donation amounts
- **Geographic Mapping:** Map donation origins by ZIP code
- **Time Series Analysis:** Analyze historical donation trends and forecast future donations

## Current Focus — Week 1
Load the dataset(s), conduct preliminary EDA, and perform **safe standardization** (column naming, formatting, type conversion) to assess data quality and prepare for donor matching and consolidation.

Week 1 deliverable: EDA summary + initial data quality notes + recommended next steps.

# Week 1 - Load + Prelimenary EDA (BGC Donor DATA)

## Objective

- Load dataset
- Confirm columns + data types
- Check missing values (email, address, etc)
- Check duplicates
- Write initial data quality notes

## Data Handling rules

- Raw is read-only
- Working copy is for edits
- Batch cleaning + validation after each batch
- No guessing missing values in Week 1

## Step 1 — Load Consolidated CSV (Working Copy)

Purpose:
Load the consolidated donor dataset for preliminary EDA.

Rules:
- Raw file stays unchanged
- I create a working dataframe for analysis
- No permanent cleaning decisions yet (Week 1 = assess + document)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.exists(), RAW_DIR

(True,
 PosixPath('/home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project/data/raw'))

In [3]:
[p.name for p in RAW_DIR.glob("*.csv")]

['2019-2026.xlsx - CSV for Boys & Girls Clubs of B.csv']

In [4]:
# Load the only CSV in raw
csv_files = sorted(RAW_DIR.glob("*.csv"))
len(csv_files)

1

In [6]:
csv_path = csv_files[0]
raw = pd.read_csv(csv_path)

df = raw.copy(deep=True)

csv_path.name, df.shape

('2019-2026.xlsx - CSV for Boys & Girls Clubs of B.csv', (753, 13))

**Result:** Dataset loaded successfully.

Next:

- Confirm column names and data types

- Measure missing values (especially contact fields)

- Check duplicates (record-level)


## Step 2 — Dataset Structure (Shape, Columns, Data Types)

**Purpose:**
Understand the size of the dataset and how fields are represented.

**Key question:**
Are columns stored in the correct type (numbers as numeric, dates as datetime, etc.)?


In [8]:
df.shape

(753, 13)

**Interpretation (Baseline):** The dataset contains 753 rows and 13 columns.  

**Next step:** Review column names and data types to identify key donor identity fields and donation fields.

In [9]:
df.columns

Index(['Donation Date (UTC)', 'Intended Donation', 'Amount Charged',
       'First Name', 'Last Name', 'Email', 'Address', 'City', 'State',
       'Zipcode', 'Phone', 'Campaign', 'Comment'],
      dtype='object')

In [10]:
df.info

<bound method DataFrame.info of     Donation Date (UTC) Intended Donation Amount Charged     First Name  \
0             3/14/2019          $100.00        $100.00           Bruce   
1             3/15/2019          $200.00        $208.90         Lee Ann   
2             3/18/2019          $100.00        $100.00   Paul & Nicole   
3              4/6/2019          $100.00        $104.60          Denise   
4             6/10/2019          $650.00        $650.00          George   
..                  ...               ...            ...            ...   
748          12/29/2025          $500.00        $532.00            Leah   
749          12/29/2025          $100.00        $106.65            Mark   
750          12/29/2025          $100.00        $106.65          Marian   
751          12/31/2025          $250.00        $266.16            R J    
752           1/16/2026           $50.00         $53.48            Bart   

       Last Name                       Email       Address         

**Notes:**

- `shape` = (753 rows, 13 columns)

- `non-null` = number of rows that are NOT missing for that column

- If `non-null` is less than total rows → that column has missing values


## Step 3 — Missing Values Snapshot (Overall + Contact Fields)

**Purpose:**

Quantify missingness to understand data quality risks (e.g., inability to contact donors).

**Interpretation rule:**

Higher missing % in email/address/phone increases risk for donor communication + retention.


In [11]:
# overall missingness top 15
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.head(15)

Phone                  0.993360
Address                0.970784
State                  0.965471
City                   0.965471
Zipcode                0.917663
Comment                0.903054
Email                  0.467463
Campaign               0.223108
Last Name              0.103586
First Name             0.000000
Amount Charged         0.000000
Intended Donation      0.000000
Donation Date (UTC)    0.000000
dtype: float64

In [12]:
# Only columns that include "email/phone/address/zip" 
contact_cols = [c for c in df.columns if any(k in c.lower() for k in ["email", "phone", "address", "zip"])]
contact_cols

['Email', 'Address', 'Zipcode', 'Phone']

In [13]:
df[contact_cols].isna().mean().sort_values(ascending=False)

Phone      0.993360
Address    0.970784
Zipcode    0.917663
Email      0.467463
dtype: float64

**Data quality note:** Missing contact fields can lead to:

- Donation loss (no follow-up possible)

- Lower engagement (cannot notify donors about events/impact)

- Recognition issues (thank-you messages, receipts)


## Step 4 — Duplicate Records (Preliminary)

**Purpose:**
Check if exact duplicate rows exist in the consolidated dataset.

**Note:**
This does NOT yet detect “same donor entered differently” — that comes later with donor matching rules.


In [14]:
df.duplicated().sum()

2

In [15]:
df = df.drop_duplicates()
df.shape

(751, 13)

**Result:** Exact duplicates were identified and removed.

Next:

Design donor identity rules (email/phone/name+zip) and build a possible-duplicates review list.

## Step 5 — Identity Fields Inventory (Donor Matching Readiness)

**Purpose:**
Identify which fields can be used to uniquely identify donors and how complete they are.

**Focus:**
email, phone, name, address, zip (and any donor_id if present)

In [17]:
identity_cols = [c for c in df.columns if any(k in c.lower() for k in ["email","phone","name","address","zip","donor","id"])]
identity_cols

['First Name', 'Last Name', 'Email', 'Address', 'Zipcode', 'Phone']

**Notes:**

- These fields will drive donor matching priority.

- High missingness or inconsistent formatting increases dedupe workload.


## Step 6 — Identity Field Strength Assessment

For each identity field (Email, Phone):

1) Missing %
   df["Email"].isna().mean()

2) Unique non-missing count
   df["Email"].nunique(dropna=True)

3) Count of repeated values
   (df["Email"].value_counts(dropna=True) > 1).sum()

**Interpretation:**

- Low missing + high unique = strong identity field
- High missing = weak identity field
- Many repeats = heavy matching needed

## Step 6A — Check missing percentage

In [18]:
df["Email"].isna().mean()

0.4673768308921438

In [19]:
df["Phone"].isna().mean()

0.9933422103861518

## Identity Field Strength (Week 1 Findings)

**Email:**
- ~46% missing

- Moderate reliability for donor matching

**Phone:**
- ~ 99% missing

- Weak identity field (cannot rely on it alone)

**Implication:**

Donor matching should prioritize:

1) Email

2) Name + Zip (fallback)

3) Phone (low priority)

**Conclusion:**

No single strong donor ID exists → multi-field matching required in Week 2.

## Step 6B — Check uniqueness

In [20]:
df["Email"].nunique(dropna=True)

215

In [21]:
df["Phone"].nunique(dropna=True)

5

## Identity Field Pattern (Week 1)

**Email:**

- Missingness is moderate (~half missing)

- Unique emails much smaller than non-missing count → many repeats

- Interpretation: repeat donors, duplicates, or shared emails likely

**Phone:**

- Missingness is high (>70% missing)

- Unique phones close to non-missing count → phones mostly unique when present

- Interpretation: phone can support matching but cannot be primary


In [22]:
(df["Email"].value_counts(dropna=True) > 1).sum()

36

In [23]:
(df["Phone"].value_counts(dropna=True) > 1).sum()

0

## Week 1 EDA Wrap-Up (Donor Matching Readiness)

**Key findings:**

- Email is moderately complete but repeats often → likely repeat donations, duplicate entries, or shared emails.

- Phone is highly incomplete (>70% missing) → cannot be a primary donor identifier.

- Next step (Week 2): build donor matching rules prioritizing Email, then Name+Zip/Address as fallback, with “do no harm” dedupe + mentor/stakeholder review.


## Open Questions for Mentor (to confirm before merging)

1) What does “Paid = X” represent (registration fee, ticket, donation, or mixed)?

2) Should registration-only people be included as donor contacts?

3) Donor identity priority: Email vs Name+Zip vs Phone — confirm preferred matching rules.

4) Household rule: same address with multiple names — merge or keep separate?


**Response:** 

1. Compare the contacts list to the donors, how many are repeated donors..Anything I have questions ignore it.

2. Registration files skip merging. 

3. Great futures donation log and the Great futures 2024 and 2025 put in the same table and see how they relate to the contacts.

4. Check for contacts for all donors doesn't matter which form of contact I use. 